# HG gain calibration from finger plots

Extracts the single-photoelectron peak spacing from CITIROC finger spectra
and reports how it varies with the preamplifier HG register code.

**Pipeline**

`discover HG folders -> read CSV -> normalise -> find peaks -> keep stable
region -> fit ADC vs peak number -> spacing vs HG code`

The measured quantity is **ADC bins per photoelectron**, which is the product
of the SiPM gain, the preamplifier gain, the shaper response and the ADC
scale. It is not the preamplifier gain \(A_v\) on its own.

Author: Muhammad Abdullahi (muhammad.abdullahi@gssi.it)


## Imports


In [ ]:
import os
import re
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.optimize import curve_fit


## Configuration

Paths are relative to this notebook. Put the acquisition folders under
`_pwd`, or point `_pwd` at wherever they live. Nothing below refers to an
absolute path.

Each acquisition folder is expected to carry its HG code in its name, for
example `Updated_HG=47_offset=6950_High=3.6V/`, and to contain `data.csv`.


In [ ]:
_pwd        = './data/gain_calibration/'   # acquisition folders live here
plt_folder  = './Results'                  # figures are written here
csv_name    = 'data.csv'                   # file inside each folder

# Channel under test
DAQ, ASIC, CH, GAIN = 1, 'A', 30, 'HG'

# Peak finding.  height is a FRACTION of each spectrum's maximum, so the
# threshold scales with the data instead of losing peaks at high HG.
height     = 0.02
distance   = 20
tolerance  = 0.15

# Display
size    = 14
xlim    = (-20, 450)
savefig = True

os.makedirs(plt_folder, exist_ok=True)


## Channel mapping

Column order in the CSV, 640 columns in total:

| range | contents |
|---|---|
| 0-159   | DAQ1, ASIC A..E, LG, CH0..31 |
| 160-319 | DAQ1, ASIC A..E, HG, CH0..31 |
| 320-479 | DAQ2, ASIC A..E, LG, CH0..31 |
| 480-639 | DAQ2, ASIC A..E, HG, CH0..31 |


In [ ]:
ASIC_ORDER = ['A', 'B', 'C', 'D', 'E']


def col_index(daq, asic, ch, gain):
    """Column number for one channel."""
    if daq not in (1, 2):
        raise ValueError('daq must be 1 or 2')
    asic = asic.upper()
    if asic not in ASIC_ORDER:
        raise ValueError(f'asic must be one of {ASIC_ORDER}')
    if not (0 <= ch <= 31):
        raise ValueError('ch must be 0..31')
    gain = gain.upper()
    if gain not in ('LG', 'HG'):
        raise ValueError("gain must be 'LG' or 'HG'")

    return ((daq - 1) * 320
            + (0 if gain == 'LG' else 160)
            + ASIC_ORDER.index(asic) * 32
            + ch)


def read_hist_csv(csv_path):
    """Histogram matrix, shape (nbins, 640)."""
    arr = pd.read_csv(csv_path, header=None).to_numpy(dtype=float)
    if arr.shape[1] == 641:          # leading bin-index column
        arr = arr[:, 1:]
    if arr.shape[1] != 640:
        raise ValueError(f'{csv_path}: expected 640 columns, '
                         f'got {arr.shape[1]}')
    return arr


def get_channel_hist(arr, daq, asic, ch, gain):
    counts = arr[:, col_index(daq, asic, ch, gain)]
    return np.arange(len(counts)), counts


## File discovery

The HG code is read from each folder name rather than assumed from list
position. Adding, removing or reordering acquisitions therefore cannot
mislabel them.


In [ ]:
def discover_acquisitions(folder, csv_name=csv_name, pattern=r'HG=(\d+)'):
    """Map HG code -> path of its CSV."""
    found = {}
    if not os.path.isdir(folder):
        raise FileNotFoundError(f'Data folder not found: {folder}')

    for name in sorted(os.listdir(folder)):
        sub = os.path.join(folder, name)
        if not os.path.isdir(sub):
            continue
        m = re.search(pattern, name)
        if not m:
            continue
        csv_path = os.path.join(sub, csv_name)
        if not os.path.isfile(csv_path):
            print(f'  skipped, no {csv_name}: {name}')
            continue
        code_value = int(m.group(1))
        if code_value in found:
            raise ValueError(f'HG={code_value} appears twice: '
                             f'{found[code_value]} and {csv_path}')
        found[code_value] = csv_path
    return found


csv_path_by_hg = discover_acquisitions(_pwd)
hg_values = sorted(csv_path_by_hg)
print(f'{len(hg_values)} acquisitions found: {hg_values}')


## Load the spectra

Each CSV is read once. Every spectrum is normalised to its own maximum, so
the peak-finding threshold is a fraction rather than an absolute count.


In [ ]:
def load_spectra(csv_path_by_hg, daq, asic, ch, gain):
    data = {}
    for hg in sorted(csv_path_by_hg):
        arr = read_hist_csv(csv_path_by_hg[hg])
        x, counts = get_channel_hist(arr, daq, asic, ch, gain)
        cmax = counts.max()
        data[hg] = (x, counts / cmax if cmax > 0 else counts)
    return data


spectra = load_spectra(csv_path_by_hg, DAQ, ASIC, CH, GAIN)
print(f'loaded {len(spectra)} spectra for DAQ{DAQ} ASIC-{ASIC} '
      f'CH{CH} ({GAIN})')


## Peak spacing

`filter_stable_region` keeps the longest run of consistent gaps, tolerating
one outlier. The spacing is then the slope of a least-squares fit of ADC
position against peak number, which is less sensitive to a single bad gap
than the mean of the gaps.


In [ ]:
def filter_stable_region(mask):
    """Longest run of True in mask, tolerating one isolated False.

    Returns the peak slice (start, stop). mask indexes gaps, and gaps
    s..e-1 involve peaks s..e, hence the +2 on the upper bound.
    """
    best = (0, 0)
    i = 0
    n = len(mask)
    while i < n:
        seg, false_count = [], 0
        for j in range(i, n):
            if mask[j]:
                seg.append(j)
                false_count = 0
            else:
                false_count += 1
                if false_count > 1:
                    break
                seg.append(j)
        while seg and not mask[seg[-1]]:      # do not end on a blip
            seg.pop()
        if seg and (seg[-1] - seg[0]) > (best[1] - best[0]):
            best = (seg[0], seg[-1])
        i = (seg[-1] + 2) if seg else (i + 1)
    return best[0], best[1] + 2


def linear(x, m, b):
    return m * x + b


def fit_spacing(x, counts_n, height=height, distance=distance,
                tolerance=tolerance, index_shift=0):
    """Peak spacing in ADC bins, from a fit of position against order."""
    peaks, _ = find_peaks(counts_n, height=height, distance=distance)
    if len(peaks) < 4:
        return None

    peak_x = x[peaks]
    diffs = np.diff(peak_x)
    median_d = np.median(diffs)
    if median_d <= 0:
        return None

    mask = np.abs(diffs - median_d) < tolerance * median_d
    if not np.any(mask):
        return None

    s, e = filter_stable_region(mask)
    px, py = peak_x[s:e], counts_n[peaks][s:e]
    if len(px) < 4:
        return None

    ini = int(np.clip(np.argmax(py) + index_shift, 0, len(px) - 1))
    adc = px[ini:]
    if len(adc) < 4:
        return None

    order = np.arange(len(adc))
    popt, pcov = curve_fit(linear, order, adc)
    resid = adc - linear(order, *popt)
    ss_tot = np.sum((adc - adc.mean()) ** 2)

    return {
        'adc': adc,
        'order': order,
        'n_peaks': len(adc),
        'spacing': float(popt[0]),
        'spacing_err': float(np.sqrt(pcov[0, 0])),
        'pedestal': float(popt[1]),
        'r2': float(1 - np.sum(resid ** 2) / ss_tot) if ss_tot > 0 else 0.0,
    }


results = {}
for hg, (x, cn) in spectra.items():
    r = fit_spacing(x, cn)
    if r is None:
        print(f'HG={hg}: not analysable')
        continue
    results[hg] = r
    print(f"HG={hg}: spacing = {r['spacing']:6.2f} +/- "
          f"{r['spacing_err']:.2f} ADC/p.e.  "
          f"({r['n_peaks']} peaks, R2 = {r['r2']:.5f})")


## Finger plots


In [ ]:
def plot_fingers(spectra, results, save=savefig):
    N = len(spectra)
    cmap = plt.get_cmap('winter')
    colors = [cmap(i / (N - 1)) for i in range(N)] if N > 1 else [cmap(0.0)]

    fig, ax = plt.subplots(figsize=(8, 4))
    for i, hg in enumerate(sorted(spectra)):
        x, cn = spectra[hg]
        ax.plot(x, cn, color=colors[i], alpha=0.6, lw=1.0, label=f'HG={hg}')
        if hg in results:
            adc = results[hg]['adc']
            ax.plot(adc, cn[adc.astype(int)], 'x', color=colors[i],
                    ms=6, mew=1.5)

    ax.set_xlim(*xlim)
    ax.set_ylim(bottom=0)
    ax.grid(True, which='major', alpha=0.3)
    ax.set_title(f'DAQ{DAQ} ASIC-{ASIC} CH{CH} ({GAIN})', fontsize=size)
    ax.set_xlabel('ADC bin', fontsize=size)
    ax.set_ylabel('Count (a.u.)', fontsize=size)
    ax.legend(ncol=2, fontsize=10)
    plt.tight_layout()

    if save:
        path = os.path.join(
            plt_folder, f'finger_DAQ{DAQ}_{ASIC}_CH{CH}_{GAIN}.pdf')
        plt.savefig(path, dpi=300)
        print(f'Plot saved: {path}')
    plt.show()
    plt.close()
    return colors


colors = plot_fingers(spectra, results)


## ADC position against peak number

Equally spaced photoelectron peaks give a straight line. Curvature here
means the peak finder is merging peaks, missing them, or the response is
compressing at high amplitude.


In [ ]:
def plot_adc_vs_order(results, colors, save=savefig):
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, hg in enumerate(sorted(results)):
        r = results[hg]
        xf = np.linspace(-1, r['order'].max() + 1, 100)
        ax.plot(xf, linear(xf, r['spacing'], r['pedestal']), '--',
                color=colors[i], alpha=0.8)
        ax.scatter(r['order'], r['adc'], color=colors[i], label=f'HG={hg}')

    ax.grid(True, which='major', alpha=0.3)
    ax.set_title(f'DAQ{DAQ} ASIC-{ASIC} CH{CH} ({GAIN})', fontsize=size)
    ax.set_xlabel('Peak number', fontsize=size)
    ax.set_ylabel('ADC bin', fontsize=size)
    ax.legend(ncol=2, fontsize=10)
    plt.tight_layout()

    if save:
        path = os.path.join(
            plt_folder, f'adc_vs_order_DAQ{DAQ}_{ASIC}_CH{CH}_{GAIN}.pdf')
        plt.savefig(path, dpi=300)
        print(f'Plot saved: {path}')
    plt.show()
    plt.close()


plot_adc_vs_order(results, colors)


## Spacing against HG code

Two models are fitted. If \(A_v = C_\mathrm{in}/C_f\) with \(C_f\) linear
in the register code, the spacing follows \(1/(\alpha + \beta n)\) rather
than a straight line, and \(-\alpha/\beta\) gives the code at which
\(C_f\) would vanish.

Note that a hyperbola over a short span of codes is close to straight, so a
good linear fit is not by itself evidence of linearity.


In [ ]:
def reciprocal(x, alpha, beta):
    return 1.0 / (alpha + beta * x)


def r_squared(y, pred):
    ss_tot = np.sum((y - y.mean()) ** 2)
    return float(1 - np.sum((y - pred) ** 2) / ss_tot) if ss_tot > 0 else 0.0


def plot_spacing_vs_code(results, save=savefig):
    hg = np.array(sorted(results), dtype=float)
    sp = np.array([results[h]['spacing'] for h in sorted(results)])
    er = np.array([results[h]['spacing_err'] for h in sorted(results)])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.errorbar(hg, sp, yerr=er, fmt='o', capsize=3, label='measured')

    if len(hg) >= 3:
        grid = np.linspace(hg.min() - 0.5, hg.max() + 0.5, 300)

        pl, _ = curve_fit(linear, hg, sp)
        r2_lin = r_squared(sp, linear(hg, *pl))
        ax.plot(grid, linear(grid, *pl), '--',
                label=f'linear, $R^2$={r2_lin:.4f}')

        seed, _ = curve_fit(linear, hg, 1.0 / sp)
        pr, _ = curve_fit(reciprocal, hg, sp, p0=[seed[1], seed[0]],
                          maxfev=20000)
        r2_rec = r_squared(sp, reciprocal(hg, *pr))
        ax.plot(grid, reciprocal(grid, *pr), '-',
                label=r'$1/(\alpha+\beta n)$, ' + f'$R^2$={r2_rec:.4f}')

        print(f'linear      R2 = {r2_lin:.5f}')
        print(f'reciprocal  R2 = {r2_rec:.5f}')
        print(f'preferred      = '
              f"{'reciprocal' if r2_rec > r2_lin else 'linear'}")
        print(f'C_f extrapolates to zero at code {-pr[0] / pr[1]:.1f}')

    ax.grid(True, which='major', alpha=0.3)
    ax.set_title(f'DAQ{DAQ} ASIC-{ASIC} CH{CH} ({GAIN})', fontsize=size)
    ax.set_xlabel('Preamplifier HG code', fontsize=size)
    ax.set_ylabel('Spacing (ADC bins / p.e.)', fontsize=size)
    ax.set_xticks(sorted(results))
    ax.legend(fontsize=10)
    plt.tight_layout()

    if save:
        path = os.path.join(
            plt_folder, f'spacing_vs_code_DAQ{DAQ}_{ASIC}_CH{CH}_{GAIN}.pdf')
        plt.savefig(path, dpi=300)
        print(f'Plot saved: {path}')
    plt.show()
    plt.close()


plot_spacing_vs_code(results)


## Save the numbers

The fitted spacings are written alongside the figures so the plots can be
regenerated or compared without re-running the analysis.


In [ ]:
summary = {
    'daq': DAQ, 'asic': ASIC, 'channel': CH, 'gain': GAIN,
    'height': height, 'distance': distance, 'tolerance': tolerance,
    'results': {
        str(hg): {k: v for k, v in r.items()
                  if k not in ('adc', 'order')}
        for hg, r in results.items()
    },
}

out = os.path.join(plt_folder,
                   f'spacing_DAQ{DAQ}_{ASIC}_CH{CH}_{GAIN}.json')
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Results saved: {out}')
